# Music genre classification
The goal of this project is to extract the necessary features from the raw audio files, and use them to train the kNN, ANN and CNN models with it. After that, it could be compared to models that were trained with the dataset provided CSV files. Only 30 v 30 or also v 3?
- extracted audio features manually ✔     
- train kNN on extracted features   ✔       
    - train kNN on provided features ✔     
    - train kNN on provided features (3s) ✔
- train ANN on extracted features
    - train ANN on provided features
    - train ANN on provided features (3s)
- train CNN on provided spectrograms

### Importing GTZAN dataset

In [3]:
import os # to interact with OS: file/directory opeartions, envoronmental variables...
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"        
os.environ["MKL_NUM_THREADS"] = "1"        
os.environ["NUMEXPR_NUM_THREADS"] = "1"   # these 4 were added to prevent the python kernel from dying when
import pandas as pd # just, used to import the csvs. dataframe manipulation (csv/excel)

In [4]:
# Path to Dataset
PATH = "/home/ket/Documents/gtzan/archive/Data/genres_original"
PATH_CSV_30 = "/home/ket/Documents/gtzan/archive/Data/features_30_sec.csv"
PATH_CSV_3 = "/home/ket/Documents/gtzan/archive/Data/features_3_sec.csv"
PATH_IMAGES = "/home/ket/Documents/gtzan/archive/Data/images_original"
MFCC_N = 14 # How may MFCC to keep (from 20)

In [5]:
# Import CSV and look at it
df_30 = pd.read_csv(PATH_CSV_30)
df_3 = pd.read_csv(PATH_CSV_3)
df_30.head(3)
df_30.tail(3)
# df_30.describe
#df_30.columns # 60 COLUMNS
# pandas quickstart: https://github.com/techwithtim/PanadasTutorial

,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
997,rock.00097.wav,661794,0.432142,0.075268,0.081651,0.000322,2077.526598,231657.968040,1927.293153,74717.124394,...,33.597008,-12.845291,36.367264,3.440978,36.001110,-12.588070,42.502201,-2.106337,29.865515,rock
998,rock.00098.wav,661794,0.362485,0.091506,0.083860,0.001211,1398.699344,240318.731073,1818.450280,109090.207161,...,46.324894,-4.416050,43.583942,1.556207,34.331261,-5.041897,47.227180,-3.590644,41.299088,rock
999,rock.00099.wav,661794,0.358401,0.085884,0.054454,0.000336,1609.795082,422203.216152,1797.213044,120115.632927,...,59.167755,-7.069775,73.760391,0.028346,76.504326,-2.025783,72.189316,1.155239,49.662510,rock


In [7]:
# Pregled podatkov, any null values? 
# df_30.isnull().sum() # no null values
# df_30.info()

It appears that no data is missing
# Data preprocessing
The data set was imported, now it has to be determined which of the contained data, or rather clomuns, are relevant. There's 60 columns, let's take a look.
### What is needed
`filename` is not needed, can be removed, `label` has the genre . These three are needed because of the project description I wrote: MFCCs, ZCR, and Chroma features. But I see there's other features I will require like Spectral Centroids, Tempo and RMS (Root Mean Square).
Also do I need both avg and means? yea
### What is useful
`length`, `chroma_stft_mean`, `chroma_stft_var`, `rms_mean`
`rms_var`, `spectral_centroid_mean`, `spectral_centroid_var`,
`spectral_bandwidth_mean`, `spectral_bandwidth_var`, `rolloff_mean`,
`rolloff_var`, `zero_crossing_rate_mean`, `zero_crossing_rate_var`,
`harmony_mean`, `harmony_var`, `perceptr_mean`, `perceptr_var`, `tempo`
### What is not nedeed
`filename` and probably we could do without all the MFCCs. I remember 13 being enough.

In [8]:
# This code is reused below
# MFCC_N = 20
# col = df_30.pop("label")
# df_30.insert(0, "label", col)
# df_30.drop("filename", inplace=True, axis=1)
# # this loop removes as many MFCCs as defined at the top
# for n in range(20-MFCC_N):
#     name1 = f"mfcc{20-n}_var"
#     name2 = f"mfcc{20-n}_mean"
#     # print(name1, " and ", name2)
#     df_30.drop(name1, inplace=True, axis=1)
#     df_30.drop(name2, inplace=True, axis=1)
# df_30.columns



## Split Train & Test kNN - 30s

In [9]:
# ARCHIVED
# # https://www.geeksforgeeks.org/machine-learning/how-to-split-a-dataset-into-train-and-test-sets-using-python/
# # https://pylearnai.com/machine-learning/csv-to-prediction-machine-learning-python/
# # https://gao-hongnan.github.io/gaohn-galaxy/machine_learning/neighbours/k_nearest_neighbours/knn_feature_scaling.html 25.6.26
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.neighbors import KNeighborsClassifier

# ### variables
# MFCC_N = 20
# K_N = 30
# TEST_SIZE = [0.1, 0.2, 0.3, 0.4]
# RANDOM_STATE = 100
# ###

# #omega loop to collect all the possible variations
# for i in range(MFCC_N):
#     for ii in range(K_N):
#         for iii in TEST_SIZE:
#             for iiii in range(RANDOM_STATE):

#                 col = df_30.pop("label")
#                 df_30.insert(0, "label", col)
#                 df_30.drop("filename", inplace=True, axis=1)
#                 # this loop removes as many MFCCs as defined at the top
#                 for n in range(20-MFCC_N):
#                     name1 = f"mfcc{20-n}_var"
#                     name2 = f"mfcc{20-n}_mean"
#                     # print(name1, " and ", name2)
#                     df_30.drop(name1, inplace=True, axis=1)
#                     df_30.drop(name2, inplace=True, axis=1)

#                 X = df_30.drop(columns=["label"])
#                 y = df_30['label'] # by genre
#                 # NORMALIZATION - we need this because "The K-NN algorithm relies on the Euclidean
#                 # distance between data points, making it highly sensitive to the scale of 
#                 # the features. If there’s a discrepancy in the scale across different features,
#                 # the feature with the larger scale will overshadow the others, leading to 
#                 # biased predictions."
#                 scale = StandardScaler()
#                 X_scaled = scale.fit_transform(X)

#                 X_train, X_test, y_train, y_test = train_test_split(
#                     X_scaled, y,
#                     test_size=iii, # 70 train 30 test
#                     random_state=iiii, # idk, tutorial had 42
#                     stratify=y # such that 30 songs from each genre are used for testing
#                     )

#                 # KNN
#                 knn = KNeighborsClassifier(n_neighbors=K_N).fit(X_train, y_train) # For MFCC = 20, K = 10 gives the best accuracy : 0.6767
#                 y_test_normalized_preds = knn.predict(X_test)                    # For MFCC = 13, K = 10 gives the best accuracy : 0.7134
#                 print("KNN accuracy:", knn.score(X_test, y_test))                # For MFCC = 14, K = 10 gives the best accuracy : 0.68
#                 print()


#                 # writing it down so I dont need to do it manually
#                 CSV_PATH_KNN_30 = "knn-data-df-30.csv"
#                 features = {
#                     "MFCC_N": [i],
#                     "TEST_SIZE": [iii],
#                     "RANDOM_STATE": [iiii],
#                     "K_N": [ii],
#                     "ACCURACY": [knn.score(X_test, y_test)]
#                 }

#                 if os.path.exists(CSV_PATH_KNN_30):
#                     df = pd.read_csv(CSV_PATH_KNN_30)
#                     df = pd.concat([df, pd.DataFrame(features)], ignore_index=True)
#                 else:
#                     df = pd.DataFrame(features)
#                 df.to_csv(CSV_PATH_KNN_30, index=False)

#                 print(df)


In [10]:
# COMMENTED TO NOT RUN EVERY TIME
# # https://www.geeksforgeeks.org/machine-learning/how-to-split-a-dataset-into-train-and-test-sets-using-python/
# # https://pylearnai.com/machine-learning/csv-to-prediction-machine-learning-python/
# # https://gao-hongnan.github.io/gaohn-galaxy/machine_learning/neighbours/k_nearest_neighbours/knn_feature_scaling.html 25.6.26
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.neighbors import KNeighborsClassifier
# from tqdm import tqdm

# ### variables
# MFCC_N = 20
# K_N = 30
# TEST_SIZE = [0.1, 0.2, 0.3, 0.4]
# RANDOM_STATE = [0, 42, 99]
# CSV_PATH_KNN_30 = "knn-data-df-30.csv"
# ###


# #omega loop to collect all the possible variations
# results = []

# df_base = df_30.copy()
# col = df_base.pop("label")
# df_base.insert(0, "label", col)
# df_base.drop("filename", inplace=True, axis=1)

# for i in tqdm(range(1, MFCC_N+1), desc="kNN Data collection progress"):       
#     print("We're at MFCC: ", MFCC_N)
#     df_work = df_base.copy()

#     for n in range(20 - i):
#         df_work.drop(f"mfcc{20-n}_var", inplace=True, axis=1)
#         df_work.drop(f"mfcc{20-n}_mean", inplace=True, axis=1)

#     X = df_work.drop(columns=["label"])
#     y = df_work["label"]
#     X_scaled = StandardScaler().fit_transform(X)

#     for k in tqdm(range(1, K_N+1), desc="K-Neighbors", leave=False):      
#         for test_size in tqdm(TEST_SIZE, desc="Test size", leave=False):
#             for seed in tqdm(RANDOM_STATE, desc="seed", leave=False): 
                
#                 # col = df_work.pop("label")
#                 # df_work.insert(0, "label", col)
#                 # df_work.drop("filename", inplace=True, axis=1)        

#                 X_train, X_test, y_train, y_test = train_test_split(
#                     X_scaled, y, test_size=test_size, random_state=seed, stratify=y
#                 )

#                 knn = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
#                 acc = knn.score(X_test, y_test)

#                 results.append({"MFCC_N": i, "K_N": k, "TEST_SIZE": test_size,
#                                  "RANDOM_STATE": seed, "ACCURACY": acc})


# pd.DataFrame(results).to_csv(CSV_PATH_KNN_30, index=False)


After brief manual testing, the K that gave the best results was `K=10`, specifically with `MFCC_N=13`.
To the end of finding a combination of parametets that yield a higher accuracy, I automated some data collection - will be used later.
These were the parameters and their range of values:
`MFCC_N = [1-20]`
`K_N = [1-30]`
`TEST_SIZE = [0.1, 0.2, 0.3, 0.4]`
`RANDOM_STATE = [0, 42, 99]`


In [6]:
df_knn = pd.read_csv("/home/ket/Desktop/Jupyter ML/knn-data-df-30.csv")
df_knn_sorted = df_knn.sort_values(by="ACCURACY", ascending=False)

df_knn_sorted.head(10)

,MFCC_N,K_N,TEST_SIZE,RANDOM_STATE,ACCURACY
6840,20,1,0.1,0,0.770
4717,14,4,0.1,42,0.770
4729,14,5,0.1,42,0.760
5092,15,5,0.2,42,0.755
4381,13,6,0.1,42,0.750
6864,20,3,0.1,0,0.750
3973,12,2,0.1,42,0.750
6206,18,8,0.1,99,0.750
4741,14,6,0.1,42,0.750
5089,15,5,0.1,42,0.750


## Feature extraction
In the project description I wrote that I would extract the needed features manually. To achieve that I will use the `AudioFeaturizer` library. The out put data is not 1:1 as the data provided by the dataset but it should do.
Crucial to note that one of the original audio files seems to be corrupted as it is simply not working, and there is nothing that can be extracted from it. The file in question is `jazz.00054.wav`.

In [7]:
# https://pypi.org/project/AudioFeaturizer/
from AudioFeaturizer.audio_featurizer import *
# just testing 
test =audio_process("/home/ket/Documents/gtzan/archive/Data/genres_original/jazz/jazz.00055.wav")
print(type(test))
genres = os.listdir(PATH)
print("Genres: ", genres)
results = []

# COMMENTED SO IT DOESNT EXTRACT FEATURES EVERY TIME
for genre in tqdm(genres, desc="Feature exctraction"):
    genre_path = os.path.join(PATH, genre)
    for filename in os.listdir(genre_path):
        if(filename != "jazz.00054.wav"): # broken
            file_path = os.path.join(genre_path, filename)

            try:
                features = audio_process(file_path)
                row = features.iloc[0].to_dict()
                row["genre"] = genre
                row["filename"] = filename
                results.append(row)
            except Exception as e:
                print("[Error]: Something went wrong")

extr_df = pd.DataFrame(results)
extr_df.to_csv("extracted-features.csv", index=False)


/home/ket/Desktop/Jupyter ML/tf_env/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

The audio features have now been extracted. As for the spectrograms, according to the project description the ones provided by the dataset will be used.
## Split Train & Test kNN - (extracted)

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from tqdm import tqdm

### variables
MFCC_N = 20
K_N = 30
TEST_SIZE = [0.1, 0.2, 0.3, 0.4]
RANDOM_STATE = [0, 42, 99]
CSV_PATH_KNN_EXTR = "knn-extracted-features.csv"
###


#omega loop to collect all the possible variations
results = []

df_base = extr_df.copy()
col = df_base.pop("genre")
df_base.insert(0, "genre", col)
df_base.drop("filename", inplace=True, axis=1)
#

for i in tqdm(range(1, MFCC_N+1), desc="kNN Data collection progress"):       
    print("We're at MFCC: ", MFCC_N)
    df_work = df_base.copy()

    for n in range(20 - i):
        df_work.drop(f"mfcc{20-n}", inplace=True, axis=1)

    X = df_work.drop(columns=["genre"])
    y = df_work["genre"]
    X_scaled = StandardScaler().fit_transform(X)

    for k in tqdm(range(1, K_N+1), desc="K-Neighbors", leave=False):      
        for test_size in tqdm(TEST_SIZE, desc="Test size", leave=False):
            for seed in tqdm(RANDOM_STATE, desc="seed", leave=False): 
                
                # col = df_work.pop("label")
                # df_work.insert(0, "label", col)
                # df_work.drop("filename", inplace=True, axis=1)        

                X_train, X_test, y_train, y_test = train_test_split(
                    X_scaled, y, test_size=test_size, random_state=seed, stratify=y
                )

                knn = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
                acc = knn.score(X_test, y_test)

                results.append({"MFCC_N": i, "K_N": k, "TEST_SIZE": test_size,
                                 "RANDOM_STATE": seed, "ACCURACY": acc})


pd.DataFrame(results).to_csv(CSV_PATH_KNN_EXTR, index=False)


NameError: name 'extr_df' is not defined

In [8]:
df_knn_ext = pd.read_csv("/home/ket/Desktop/Jupyter ML/knn-extracted-features.csv")
df_knn_ext_sorted = df_knn.sort_values(by="ACCURACY", ascending=False)

df_knn_ext_sorted.head(10)

,MFCC_N,K_N,TEST_SIZE,RANDOM_STATE,ACCURACY
6840,20,1,0.1,0,0.770
4717,14,4,0.1,42,0.770
4729,14,5,0.1,42,0.760
5092,15,5,0.2,42,0.755
4381,13,6,0.1,42,0.750
6864,20,3,0.1,0,0.750
3973,12,2,0.1,42,0.750
6206,18,8,0.1,99,0.750
4741,14,6,0.1,42,0.750
5089,15,5,0.1,42,0.750


It would appear that the missing mean/avg columns in the extracted features does not impact accuracy at all, in fact the results are identical.
## Split Train & Test kNN - 3s

In [ ]:
# ~22 minutes
# ### variables
# MFCC_N = 20
# K_N = 30
# TEST_SIZE = [0.1, 0.2, 0.3, 0.4]
# RANDOM_STATE = [0, 42, 99]
# CSV_PATH_KNN_3 = "knn-data-df-3.csv"
# # ###


# # #omega loop to collect all the possible variations
# results = []

# df_base = df_3.copy()
# col = df_base.pop("label")
# df_base.insert(0, "label", col)
# df_base.drop("filename", inplace=True, axis=1)

# for i in tqdm(range(1, MFCC_N+1), desc="kNN Data collection progress"):       
#     print("We're at MFCC: ", MFCC_N)
#     df_work = df_base.copy()

#     for n in range(20 - i):
#         df_work.drop(f"mfcc{20-n}_var", inplace=True, axis=1)
#         df_work.drop(f"mfcc{20-n}_mean", inplace=True, axis=1)

#     X = df_work.drop(columns=["label"])
#     y = df_work["label"]
#     X_scaled = StandardScaler().fit_transform(X)

#     for k in tqdm(range(1, K_N+1), desc="K-Neighbors", leave=False):      
#         for test_size in tqdm(TEST_SIZE, desc="Test size", leave=False):
#             for seed in tqdm(RANDOM_STATE, desc="seed", leave=False): 
                
#                 # col = df_work.pop("label")
#                 # df_work.insert(0, "label", col)
#                 # df_work.drop("filename", inplace=True, axis=1)        

#                 X_train, X_test, y_train, y_test = train_test_split(
#                     X_scaled, y, test_size=test_size, random_state=seed, stratify=y
#                 )

#                 knn = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
#                 acc = knn.score(X_test, y_test)

#                 results.append({"MFCC_N": i, "K_N": k, "TEST_SIZE": test_size,
#                                  "RANDOM_STATE": seed, "ACCURACY": acc})


# pd.DataFrame(results).to_csv(CSV_PATH_KNN_3, index=False)

In [9]:
df_knn_3 = pd.read_csv("/home/ket/Desktop/Jupyter ML/knn-data-df-3.csv")
df_knn_3_sorted = df_knn.sort_values(by="ACCURACY", ascending=False)

df_knn_3_sorted.head(10)

,MFCC_N,K_N,TEST_SIZE,RANDOM_STATE,ACCURACY
6840,20,1,0.1,0,0.770
4717,14,4,0.1,42,0.770
4729,14,5,0.1,42,0.760
5092,15,5,0.2,42,0.755
4381,13,6,0.1,42,0.750
6864,20,3,0.1,0,0.750
3973,12,2,0.1,42,0.750
6206,18,8,0.1,99,0.750
4741,14,6,0.1,42,0.750
5089,15,5,0.1,42,0.750


Oddly enough - or maybe not - the songs being split into 3s to increase the amount of data 10 times did not change the results at all.

## CNN
Below the we are accessing the mel spectrograms of the audio files. The images need to be resized to be better suited for the CNN (150x150), and also normalize the pixels that got from 0-255 to stay in a 0-1 range.
The spectrogram of `jazz.00054.wav` is also missing.

In [38]:
# https://youtu.be/x_VrgWTKkiM?si=nk-_IoqJ3EoVed0t
# import importlib # this did not fix the tensorflow problem

import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Source - https://stackoverflow.com/a/36898903
# Posted by L. Teder, modified by community. See post 'Timeline' for change history
# Retrieved 2026-07-02, License - CC BY-SA 4.0

genres = os.listdir(PATH) # da ne spacca scatole
image_paths = []
for root, dirs, files in os.walk(PATH_IMAGES):
    for name in files:
        image_paths.append(os.path.join(root, name))

print(image_paths, "\n")


X_images = []
y_labels = []


for image_path in tqdm(image_paths):
    img = load_img(image_path, target_size=(150,150)) # we shrink the image to consistent size for the CNN
    img_array = img_to_array(img)
    img_array = img_array / 255.0 # we need to scale pixels to be [0-1]
    X_images.append(img_array)
    y_labels.append(os.path.basename(os.path.dirname(image_path)))
        
X = np.array(X_images)
y = np.array(y_labels)

encoder = LabelEncoder()
y_enc = encoder.fit_transform(y)
y_cat = to_categorical(y_enc)



['/home/ket/Documents/gtzan/archive/Data/images_original/blues/blues00000.png', '/home/ket/Documents/gtzan/archive/Data/images_original/blues/blues00001.png', '/home/ket/Documents/gtzan/archive/Data/images_original/blues/blues00002.png', '/home/ket/Documents/gtzan/archive/Data/images_original/blues/blues00003.png', '/home/ket/Documents/gtzan/archive/Data/images_original/blues/blues00004.png', '/home/ket/Documents/gtzan/archive/Data/images_original/blues/blues00005.png', '/home/ket/Documents/gtzan/archive/Data/images_original/blues/blues00006.png', '/home/ket/Documents/gtzan/archive/Data/images_original/blues/blues00007.png', '/home/ket/Documents/gtzan/archive/Data/images_original/blues/blues00008.png', '/home/ket/Documents/gtzan/archive/Data/images_original/blues/blues00009.png', '/home/ket/Documents/gtzan/archive/Data/images_original/blues/blues00010.png', '/home/ket/Documents/gtzan/archive/Data/images_original/blues/blues00011.png', '/home/ket/Documents/gtzan/archive/Data/images_orig

  0%|          | 0/999 [00:00<?, ?it/s]

100%|██████████| 999/999 [00:03<00:00, 261.61it/s]


### Train/test split

In [39]:
TEST_SIZE = [0.1, 0.2, 0.3, 0.4]
RANDOM_STATE = [0, 42, 99]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y_enc
)

### Build model

In [41]:
# image augmentation - technique to stop from overfitting
model = models.Sequential([
    #convolute
    tf.keras.layers.Conv2D(64, (3,3), activation="relu", input_shape=(150,150,3)), # 3 is the number of channels (rgb) - (1 would be monochrome)
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(64, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(128, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(128, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),    
    # flatten results
    tf.keras.layers.Flatten(),
    # tf.keras.layers.Dropout(0.5) # also to help with 
    tf.keras.layers.Dense(512, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax") # softmax layer, each neuron (10) lights up with a probability
])


/home/ket/Desktop/Jupyter ML/tf_env/lib64/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Compiling model

In [43]:
model.compile(optimizer='adam', # try "rmsprop"
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 148, 148, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 74, 74, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 72, 72, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 15, 15, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 512)            │     3,211,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,477,066 (13.26 MB)

 Trainable params: 3,477,066 (13.26 MB)

 Non-trainable params: 0 (0.00 B)

### Training model & evaluating model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1)

loss, acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {acc:.4f}")

Epoch 1/10


W0000 00:00:1783162527.405822  869263 cpu_allocator_impl.cc:82] Allocation of 215730000 exceeds 10% of free system memory.
W0000 00:00:1783162533.175374  876266 cpu_allocator_impl.cc:82] Allocation of 358875136 exceeds 10% of free system memory.
W0000 00:00:1783162533.546758  876266 cpu_allocator_impl.cc:82] Allocation of 89718784 exceeds 10% of free system memory.


13/13 ━━━━━━━━━━━━━━━━━━━━ 44s 3s/step - accuracy: 0.1202 - loss: 2.3198 - val_accuracy: 0.1750 - val_loss: 2.2549
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 39s 3s/step - accuracy: 0.2416 - loss: 2.1120 - val_accuracy: 0.2150 - val_loss: 2.2181
Epoch 3/10
 1/13 ━━━━━━━━━━━━━━━━━━━━ 46s 4s/step - accuracy: 0.2031 - loss: 2.0781